# Load The datset

In [ ]:
# ============================================================
# CELL 2: LOAD REPRESENTATIVE DATASETS
# ============================================================

from datasets import load_from_disk

DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "representative_data"
)

TRAIN_PATH = os.path.join(DATA_DIR, "train")
VALIDATION_PATH = os.path.join(DATA_DIR, "validation")
TEST_PATH = os.path.join(DATA_DIR, "test")

print("=" * 70)
print("LOADING REPRESENTATIVE DATASETS")
print("=" * 70)

representative_train = load_from_disk(TRAIN_PATH)
representative_validation = load_from_disk(VALIDATION_PATH)
test_dataset = load_from_disk(TEST_PATH)

print("Train samples:", len(representative_train))
print("Validation samples:", len( representative_validation))
print("Test samples:", len(test_dataset))

print("\nColumns:")
print(train_dataset.column_names)

print("\nComplexity distribution:")
print(train_dataset["complexity"][:10])

# Dataset Formatting for SFTTrainer

## Objective

The representative training and validation datasets are converted into the prompt-completion format required for supervised fine-tuning using `SFTTrainer`.

The original dataset contains the user query, available tools, and expected tool calls in separate fields. Each example is formatted into:

- **Prompt:** System instructions, available tool definitions, and the user query
- **Completion:** The expected tool call or tool calls

The formatted datasets are used as input for LoRA fine-tuning.

In [ ]:
# ============================================================
# FORMAT REPRESENTATIVE DATASETS FOR SFT
# ============================================================

import json


def format_example(example):
    """
    Convert an xLAM dataset example into prompt-completion format
    for supervised fine-tuning.
    """

    # Parse tools and expected tool calls
    tools = json.loads(example["tools"])
    answers = json.loads(example["answers"])

    # --------------------------------------------------------
    # SYSTEM PROMPT
    # --------------------------------------------------------

    system_prompt = (
        "You are Qwen, created by Alibaba Cloud. "
        "You are a helpful assistant.\n\n"
        "# Tools\n\n"
        "You may call one or more functions to assist with the user query.\n\n"
        "You are provided with function signatures within "
        "<tools></tools> XML tags:\n"
        "<tools>\n"
    )

    # Add available tools
    for tool in tools:
        system_prompt += json.dumps(tool) + "\n"

    system_prompt += (
        "</tools>\n\n"
        "For each function call, return a json object with function "
        "name and arguments within <tool_call></tool_call> XML tags:\n"
        "<tool_call>\n"
        '{"name": <function-name>, "arguments": <args-json-object>}\n'
        "</tool_call>"
    )

    # --------------------------------------------------------
    # PROMPT
    # --------------------------------------------------------

    prompt = (
        "<|im_start|>system\n"
        + system_prompt
        + "<|im_end|>\n"
        + "<|im_start|>user\n"
        + example["query"]
        + "<|im_end|>\n"
        + "<|im_start|>assistant\n"
    )

    # --------------------------------------------------------
    # COMPLETION
    # --------------------------------------------------------

    completion = ""

    for answer in answers:

        completion += (
            "<tool_call>\n"
            + json.dumps(
                {
                    "name": answer["name"],
                    "arguments": answer["arguments"]
                }
            )
            + "\n</tool_call>\n"
        )

    completion += "<|im_end|>"

    return {
        "prompt": prompt,
        "completion": completion
    }


# ============================================================
# FORMAT REPRESENTATIVE TRAINING AND VALIDATION DATASETS
# ============================================================

print("Formatting representative datasets...\n")

train_formatted = representative_train.map(
    format_example,
    remove_columns=representative_train.column_names
)

validation_formatted = representative_validation.map(
    format_example,
    remove_columns=representative_validation.column_names
)


# ============================================================
# VERIFY FORMATTED DATASETS
# ============================================================

print("Training dataset:")
print(train_formatted)

print("\nValidation dataset:")
print(validation_formatted)

print("\nTraining example:")
print(train_formatted[0])

# SFTTrainer Configuration

## Objective

The training configuration for supervised fine-tuning is defined using `SFTConfig`.

The configuration specifies:

- Training epochs
- Batch size and gradient accumulation
- Learning rate and learning rate scheduler
- Maximum sequence length and truncation strategy
- Completion-only loss masking
- Validation frequency
- Checkpoint frequency

Completion-only loss is enabled so that the model is trained only on the expected tool-call output. The system instructions, tool definitions, and user query are excluded from the loss calculation.

In [ ]:
# ============================================================
# SFT TRAINING CONFIGURATION
# ============================================================

from trl import SFTConfig

CHECKPOINT_DIR = (
    "/content/drive/MyDrive/tool_calling_project/checkpoints/"
    "lora_representative_4000"
)

training_args = SFTConfig(

    # Output
    output_dir=CHECKPOINT_DIR,

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    # Optimization
    learning_rate=2e-4,
    lr_scheduler_type="cosine",

    # Memory
    fp16=True,
    gradient_checkpointing=True,

    # Sequence configuration
    max_length=1024,
    truncation_mode="keep_start",

    # Loss masking
    completion_only_loss=True,

    # Evaluation
    eval_strategy="steps",
    eval_steps=100,
    per_device_eval_batch_size=2,

    # Checkpoints
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,

    # Logging
    logging_strategy="steps",
    logging_steps=10,
    report_to="mlflow",

    # Reproducibility
    seed=42,

    # Dataset processing
    dataset_num_proc=2,
    packing=False,
)

print("SFTConfig created successfully!")

# LoRA Configuration and Model Preparation

## Objective

The base language model is loaded and prepared for parameter-efficient fine-tuning using LoRA (Low-Rank Adaptation).

LoRA introduces trainable low-rank adapters into selected model layers while keeping the majority of the pre-trained model parameters frozen.

The model is configured by:

- Loading the Qwen2.5-1.5B-Instruct base model and tokenizer
- Defining the LoRA configuration
- Applying LoRA adapters to the selected attention projection layers
- Inspecting the number of trainable parameters

This approach significantly reduces the number of parameters updated during fine-tuning compared to full model fine-tuning.

In [ ]:
# ============================================================
# LOAD BASE MODEL AND TOKENIZER
# ============================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from peft import (
    LoraConfig,
    get_peft_model
)


MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"


# ============================================================
# LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded successfully!")


# ============================================================
# LOAD BASE MODEL
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Base model loaded successfully!")


# ============================================================
# LORA CONFIGURATION
# ============================================================

lora_config = LoraConfig(

    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    bias="none",

    task_type="CAUSAL_LM"
)


# ============================================================
# APPLY LORA
# ============================================================

model = get_peft_model(
    model,
    lora_config
)


# ============================================================
# INSPECT TRAINABLE PARAMETERS
# ============================================================

print("\n" + "=" * 70)
print("LORA MODEL")
print("=" * 70)

model.print_trainable_parameters()

# LoRA Fine-Tuning

## Objective

The LoRA-adapted model is fine-tuned using the formatted representative training dataset.

`SFTTrainer` is initialized with:

- The LoRA-adapted model
- The SFT training configuration
- The formatted training dataset
- The formatted validation dataset
- The tokenizer

During training, the model is evaluated periodically on the validation dataset. Training metrics, validation metrics, hyperparameters, and experiment information are tracked using MLflow.

In [ ]:
# ============================================================
# INITIALIZE SFT TRAINER
# ============================================================

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_formatted,
    eval_dataset=validation_formatted,
    processing_class=tokenizer,
)

print("=" * 70)
print("SFT TRAINER CREATED SUCCESSFULLY")
print("=" * 70)

print(f"Training examples: {len(trainer.train_dataset)}")
print(f"Validation examples: {len(trainer.eval_dataset)}")


# ============================================================
# START LORA FINE-TUNING
# ============================================================

print("\n" + "=" * 70)
print("STARTING LORA FINE-TUNING")
print("=" * 70 + "\n")

train_result = trainer.train()


# ============================================================
# DISPLAY TRAINING RESULTS
# ============================================================

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

for metric, value in train_result.metrics.items():
    print(f"{metric}: {value}")

In [ ]:
# ============================================================
# SAVE FINAL LORA ADAPTER
# ============================================================

FINAL_MODEL_PATH = (
    "/content/drive/MyDrive/tool_calling_project/"
    "checkpoints/lora_representative_4000_final"
)

print("\n" + "=" * 70)
print("SAVING FINAL LORA MODEL")
print("=" * 70)

# Save the LoRA adapter
trainer.save_model(FINAL_MODEL_PATH)

# Save the tokenizer
tokenizer.save_pretrained(FINAL_MODEL_PATH)

print(f"\nFinal LoRA adapter saved successfully to:")
print(FINAL_MODEL_PATH)